# Reproducing Johnson et al. (2019) from CRCNS data set fcx-3

**Paper.** Johnson EL, King-Stephens D, Weber PB, Laxer KD, Lin JJ, Knight RT (2019). Spectral
imprints of working memory for everyday associations in the frontoparietal network. *Frontiers in
Systems Neuroscience* 12, 65. [doi:10.3389/fnsys.2018.00065](https://doi.org/10.3389/fnsys.2018.00065).
The claims file was written from the version of record (PMC6333050, open access); the paper has no
supplement.

**Data set.** CRCNS fcx-3, [doi:10.6080/K06971S9](https://doi.org/10.6080/K06971S9), read from the
public bucket `crcnsarchive` under `fcx-3/live/`. CRCNS data sets are not versioned, so the version
is recorded as `unversioned` (docs/archives.md). The paper cites no version; `live/` is the only
copy the bucket holds (there is no `old/`), with its files dated 2026-02-11 and the data
description version 0.5 of 2019-01-04, the original version. The MD5 sums of the files read, from
the data set's `checksums.md5`, are printed below, so that a later change is detectable.

**What the data set holds.** For each of the 7 subjects: the filtered continuous iEEG
(`data_primary.mat`), channel names, electrode positions, the preprocessed epochs
(`data_derived.mat`: pretrial, encoding and maintenance, and processing epochs of every channel
kept, with a per-trial table of accuracy, condition, delay and RT) and a trial table with the epoch
start samples. The docs list the analysed channels and give per-subject accuracy and RT. The
paper's MATLAB scripts (`analysis_scripts.zip`) are deposited with the data and are read here for
method details the paper leaves open (cited by file name).

**Claims** (`claims/crcns-fcx-3/10.3389-fnsys.2018.00065.yaml`), all tested:

| Claim | Tested on |
|---|---|
| table1-electrodes | channel list of the docs and the derived epochs |
| trial-counts | derived epochs and trial tables |
| fig1b-accuracy, fig1b-rt-identity-slower | docs/summary_file.csv and trial tables |
| erp-no-condition-difference | derived processing epochs |
| fig2-spatial-power, fig3-temporal-power, power-difficulty-subjects, power-low-frequencies | derived processing epochs |
| fig4-maintenance-parietal-led, fig4a-delta-theta-shift, fig4c-alpha-beta-shift, directional-pairs | derived maintenance and processing epochs |
| fig5a-delta-theta-epochs, fig5a-identity-direction, fig5a-spatiotemporal-bidirectional, fig5b-alpha-beta-no-sustained | derived processing epochs |

The data set's files are tar.gz archives (0.5 to 3.6 GB each), which cannot be read at random:
each is streamed once, decompressed in memory, reduced to what the claims use, and released. This
takes about 20 of the run's 35 minutes.

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import fcx3

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
warnings.filterwarnings("ignore", category=RuntimeWarning)

listing = fcx3.bucket_listing()
md5 = fcx3.checksums()
print(listing.to_string(index=False))
print("\nMD5 (checksums.md5) of the files read:")
for k in ["docs/summary_file.csv", "docs/summary_channel_analysis.csv", "analysis_scripts.zip"] + [f"data/{s}.tar.gz" for s in fcx3.SUBJECTS]:
    print(f"  {md5[k]}  {k}")

In [ ]:
CHAN = fcx3.channel_table()
SUMMARY = fcx3.summary_file()
META, RES = {}, {}
started = time.time()
for sid in fcx3.SUBJECTS:
    t0 = time.time()
    chan = CHAN[CHAN.subject == sid].reset_index(drop=True)
    META[sid], RES[sid] = fcx3.load_subject(sid, chan)
    print(f"{sid}: streamed and analysed in {time.time() - t0:.0f} s", flush=True)
print(f"all subjects in {(time.time() - started) / 60:.1f} min")

## Data check

Each check of the description against the values is stated under its output.

In [ ]:
rows = []
for sid in fcx3.SUBJECTS:
    m = META[sid]
    ep = m["epochs"]
    rows.append({"subject": sid, "fs": m["fs"], "primary (samples, channels)": m["primary_shape"],
                 "elec_labels": len(m["elec_labels"]), "elec_labels file": m["elec_labels_format"],
                 "derived channels": ep["data_proc"]["n_labels"], "analysed": ep["data_proc"]["shape"][1],
                 "trials (derived)": ep["data_proc"]["shape"][0], "trials (table)": len(m["trials"]),
                 "trialinfo = table": all(e["trialinfo_equals_table"] for e in ep.values()),
                 "hdr nSamples": int(ep["data_proc"]["hdr_nSamples"])})
print(pd.DataFrame(rows).to_string(index=False))
print("\nDescription: 7 subjects; S1-6 at 1 kHz, S7 at 512 Hz; derived 'fsample: 1000'; trialinfo 6 columns as listed.")
print("\nEpoch time axes (s), samples per trial, value ranges (microvolts as stored; the files give no unit):")
rows = []
for sid in fcx3.SUBJECTS:
    for k, e in META[sid]["epochs"].items():
        rows.append({"subject": sid, "epoch": k, "fs": e["fs"], "t0": e["t0"], "t1": e["t1"], "lengths": e["lengths"],
                     "starts": e["starts"], "min": round(e["min"], 1), "max": round(e["max"], 1), "sd": round(e["sd"], 2)})
print(pd.DataFrame(rows).to_string(index=False))

The description says the derived structures have `fsample: 1000`; S7's say 512, as its sampling rate.
The derived epochs carry every channel kept after preprocessing (29 to 155 per subject), of which the
docs' channel list names 192 analysed channels in all (Table 1); only those are kept here. The
continuous data (`hdr nSamples`) of the derived files have a different length from the deposited
`data_primary` in S7 only. S3's `elec_labels.mat` is a MATLAB 5 file, the others MATLAB 7.3.

In [ ]:
print("Trial tables: rows, trial numbers, accuracy, condition and jitter codes")
rows = []
for sid in fcx3.SUBJECTS:
    t = META[sid]["trials"]
    rows.append({"subject": sid, "rows": len(t), "trial numbers": f"{t.trial.min()}-{t.trial.max()}",
                 "unique": t.trial.is_unique, "correct": int(t.correct.sum()), "incorrect": int((t.correct == 0).sum()),
                 "type codes": sorted(t.type.unique()), "condition codes": sorted(t.condition.unique()),
                 "type matches condition": bool(((t.type == 1) == (t.condition == 1)).all()),
                 "jitter": sorted(t.jitter.unique()), "rt range ms": (int(t.rt.min()), int(t.rt.max())),
                 "H-G (s)": sorted(set(np.round((t.encmain_start - t.pre_start) / META[sid]["fs"], 2))),
                 "I-H-jitter (s)": sorted(set(np.round((t.proc_start - t.encmain_start) / META[sid]["fs"] - t.jitter / 1000, 2)))})
print(pd.DataFrame(rows).to_string(index=False))
print("\nTask timing (Methods, Behavioral Task): pretrial 1 s, start screen 0.8 s, fixation 0.1 s, stimuli 2 x 0.2 s,"
      " delay 0.9 or 1.15 s, cue 0.8 s.")

The description says the trial table lists each trial; in S6 and S7 it lists only the trials kept
after artifact rejection (101 and 90 of 120, numbered up to 120), and in S1 to S5 all 120. The
pretrial start (G) precedes the encoding start (H) by 1.9 s in every trial (1.87 to 1.92 s in S7,
sampled at 512 Hz), the pretrial fixation
and start screen and fixation of the Methods (1.0 + 0.8 + 0.1 s). The processing start (I) follows
H by 1.4 s plus the delay, which fits an encoding epoch of 0.6 s (the description's "600 ms + jitter")
and a 0.8-s cue.

In [ ]:
print("Derived epochs located in the continuous data (step 1: search; step 2: every analysed channel at the found offset).")
rows = []
for sid in fcx3.SUBJECTS:
    ck = META[sid]["clock"]
    ch = ck["channels"]
    num = ch.apply(lambda r: fcx3.split_label(r.channel)[1] - fcx3.split_label(r.reference)[1]
                   if fcx3.split_label(r.channel)[0] == fcx3.split_label(r.reference)[0] else None, axis=1)
    rows.append({"subject": sid, **{f"offset {k[5:]} (s)": round(v, 3) for k, v in ck["offsets_s"].items()},
                 "search r min": round(ck["search"].r.min(), 4),
                 "channels": len(ch), "r min": round(ch.r.min(), 4), "r median": round(ch.r.median(), 5),
                 "traces per channel": int(ch.n_traces.min()),
                 "reference = contact - k, k": dict(pd.Series(num).value_counts())})
print(pd.DataFrame(rows).to_string(index=False))

Each derived trace is the difference of the channel's contact and one other contact of the
continuous data (correlation at least the minimum printed, over 8 random trials and all three
epochs), so the derived channels are bipolar, labelled by one of their contacts. The reference is
the contact numbered one above or below on strips and depth leads, and the next row of the grid on
S3's 16 x 16 grid (16 below) and S6's 8 x 8 grid (8 below). The derived epochs start 1.4 s (pretrial), 1.5 s (encoding) and 1.8 s (processing)
before the trial table's columns, in every subject, so their time axes are: pretrial, 0 at G + 0.1
s; encoding, 0 at H; processing, 0 at I - 0.3 s. The deposited scripts' windows are therefore the
paper's: baseline [0.65, 0.85] s of the pretrial epoch is G + 0.75 to G + 0.95 s, 250 to 50 ms before
the start screen; maintenance [0.6, 1.5] s of the encoding epoch is 0.9 s from H + 0.6 s; processing
[0.3, 1.2] s is the 0.9 s from I. This shows that the derived data, the continuous data and the trial
table share one clock. Whether the table's columns mark the screen events is checked next.

In [ ]:
print("Timing contrast between 900-ms and 1150-ms delay trials (70-150 Hz amplitude; see fcx3.jitter_check).")
print("If the columns mark the events, the best shift is near 0 in every window; a shift beyond the shuffled maximum, with r")
print("above the shuffled maximum, means the events lie at different times relative to the column in the two groups.")
rows = []
for sid in fcx3.SUBJECTS:
    for k, v in META[sid]["jitter"].items():
        rows.append({"subject": sid, "epoch": k, "window (s)": fcx3.JITTER_WINDOWS[k], "shift 1150 vs 900 (s)": round(v["lag_s"], 3),
                     "r": round(v["r"], 2), "shuffled max |shift|": round(v["shuffled_abs_lag_max_s"], 3),
                     "shuffled max r": round(v["shuffled_r_max"], 2)})
jit = pd.DataFrame(rows)
jit["clear"] = (jit["r"] > jit["shuffled max r"]) & (jit["shift 1150 vs 900 (s)"].abs() > jit["shuffled max |shift|"])
print(jit.to_string(index=False))

The test is weak where the 70-150 Hz response is small; the column "clear" marks the windows where
the shift found exceeds every shuffled one and the alignment is better than every shuffled one.
If the processing columns of 1150-ms trials lie about 0.25 s before the events, the processing
window of those trials would hold the end of the cue. The ERP and power tests are therefore also
run on the trials with a 900-ms delay only (configuration `methods_900`), whose windows are not in
question; with about half the trials these tests have less power.

In [ ]:
grid = pd.DataFrame([{"subject": sid, "grids (dims, spacing mm, contacts)": META[sid]["grid"]} for sid in fcx3.SUBJECTS])
print(grid.to_string(index=False))
print("\nNeighbour pairs per reading of the cluster test's 'neighboring electrodes':")
print(pd.DataFrame({sid: RES[sid]["neighbours"] for sid in fcx3.SUBJECTS}).to_string())

The 'distance' reading places each bipolar channel at the midpoint of its two contacts, taking
contact n as the n-th electrode of the grid named by the label's prefix in the subject's
`_MNI_grid.csv` (S3's RH1 to RH256 are its 256-contact grid `HD_Grid`); the files do not say how
contacts are numbered on a grid, so this placement is an assumption, and with the stated 10-mm
spacings 1.5 x spacing takes in diagonal neighbours on a grid.

In [ ]:
import re

SCRIPTS = fcx3.script_files()
print("Deposited scripts (analysis_scripts.zip):", list(SCRIPTS))
pat = re.compile(r"statistic|neighbou?r|clusteralpha|correcttail|ft_prepare|montecarlo", re.I)
hits = [(f, i + 1, line.strip()) for f, t in SCRIPTS.items() for i, line in enumerate(t.splitlines()) if pat.search(line)]
print("Lines naming a statistics function, neighbours, a cluster alpha or a tail correction:", hits if hits else "none")


def show(f, *keys):
    for i, line in enumerate(SCRIPTS[f].splitlines()):
        if any(k in line for k in keys):
            print(f"  {f}:{i + 1}: {line.strip()}")


print("\nSpectral windows and padding:")
show("power_compute.m", "cfg.pad", "cfg.t_ftimwin", "cfg.toi", "cfgt.latency", "= ft_freqanalysis")
show("psi_compute.m", "cfg.pad", "cfg.toi", "cfgt.latency", "cfg.method =", "= ft_selectdata(cfgt", "subtracterp(")
print("\nERP filter and baseline:")
show("erp_compute.m", "cfg.lpf", "cfg.padding", "cfgb.", "ft_preprocessing(cfg", "nanmean", "ft_timelockanalysis(cfgf, pretrial)")
print("\nPSI and its null:")
show("subfunctions/psianalysis.m", ".*data.fourierspctrm;", "./sqrt(diag(cs(:,:,freqidx(fi)))", "psisub(:,:,freqbini) =",
     "pp = squeeze(mean(pp))", "psisub2 = squeeze(mean(psisub2))", "nulldist(:,:,permi) =", "pp(:,:,randperm(nfidx))", "psimtx(:,:,freqbini) =")
print("\nTrial selection of the time-resolved PSI:")
show("subfunctions/subtracterp.m", "cfg.trials =")

The deposited scripts settle some details and not others, as printed above. They hold no cluster
test (no FieldTrip statistics call, no neighbour definition, no cluster alpha), so the neighbour
rule and the alpha per tail are open in the paper and in the code alike; every reading is run.
power_compute.m and psi_compute.m compute the time-frequency windows over the whole derived epoch
(`cfg.toi` from its first to its last sample) and cut the processing window afterwards, so their
long windows take in data around the segment, while the Methods describe the 900-ms segments
"zero-padded to 7,500 ms". Where the Methods and the code differ, the Methods come first: the
zero-padded segment is the main reading here, and the code's whole-epoch windows are a variant.
erp_compute.m low-passes the whole epoch and subtracts the baseline mean over trials, one value per
channel, which cannot change a t-test. psianalysis.m z-scores the mean of per-trial PSI against a
null built from the trial-mean spectrum, a different statistic, and omits the conjugate of the
cross-spectrum; the PSI null used here follows the Methods' wording instead ("shuffled the
frequencies in one signal and re-computed PSI"), and the code's null is shown as a variant.

## table1-electrodes

> The analysed data comprise 192 bipolar virtual electrodes in seven subjects, 4 to 54 (mean 19) in
> PFC and 2 to 16 (mean 8) in parietal cortex per subject, as listed per subject in Table 1, forming
> 1129 same-hemisphere PFC-parietal pairs.

Counted from `docs/summary_channel_analysis.csv` (every listed channel is in the derived epochs, or
loading would have failed), with Table 1 typed from the paper for the comparison.

In [ ]:
TABLE1 = {"s1": (21, "L+R", 2, "L+R"), "s2": (24, "L+R", 8, "L+R"), "s3": (54, "R", 14, "R"), "s4": (12, "L+R", 10, "R"),
          "s5": (10, "L", 16, "L"), "s6": (8, "L", 4, "L"), "s7": (4, "L", 5, "L")}
rows = []
for sid in fcx3.SUBJECTS:
    c = CHAN[CHAN.subject == sid]
    hemi = lambda r: "+".join(sorted(c[c.region == r].hemisphere.unique()))
    rows.append({"subject": sid, "PFC": int((c.region == "PFC").sum()), "PFC hemi": hemi("PFC"),
                 "parietal": int((c.region == "parietal").sum()), "parietal hemi": hemi("parietal"),
                 "Table 1 PFC": TABLE1[sid][0], "T1 PFC hemi": TABLE1[sid][1], "Table 1 parietal": TABLE1[sid][2],
                 "T1 par hemi": TABLE1[sid][3], "pairs": len(RES[sid]["pairs"])})
t1 = pd.DataFrame(rows)
print(t1.to_string(index=False))
n_el = int(t1.PFC.sum() + t1.parietal.sum())
n_pairs = int(t1.pairs.sum())
print(f"\nanalysed channels {n_el} (paper 192); PFC per subject {t1.PFC.min()}-{t1.PFC.max()} mean {t1.PFC.mean():.2f} (paper 4-54, 19);"
      f" parietal {t1.parietal.min()}-{t1.parietal.max()} mean {t1.parietal.mean():.2f} (paper 2-16, 8)")
print(f"same-hemisphere PFC-parietal pairs {n_pairs} (paper 1129); difference {n_pairs - 1129} ({(n_pairs - 1129) / 1129:+.1%});"
      f" channels {(n_el - 192) / 192:+.1%}")
mism = t1[(t1.PFC != t1["Table 1 PFC"]) | (t1.parietal != t1["Table 1 parietal"]) | (t1["PFC hemi"] != t1["T1 PFC hemi"]) | (t1["parietal hemi"] != t1["T1 par hemi"])]
print("subjects differing from Table 1:", list(mism.subject))
s4 = t1.set_index("subject").loc["s4"]
print(f"With Table 1's 10 right parietal channels in s4, its pairs would be 6 x 10 = 60, total {n_pairs - s4.pairs + 60}.")

**Result: partially reproduced.** The data set's channel list holds 187 analysed channels, not
192: S4 has 5 right parietal channels where Table 1 gives 10; every other subject's counts and
hemispheres equal Table 1. PFC counts per subject are 4 to 54 (mean 19.00) and parietal 2 to 16
(mean 7.71), both within tolerance. The list gives 1135 same-hemisphere pairs, 6 more than the
paper's 1129 (+0.5%); with Table 1's 10 parietal channels in S4 it would give 1165. The totals
differ by no more than 10% (channels -2.6%), the criterion's partial match.

## trial-counts

> After artifact rejection subjects kept on average 96 correct trials, 32 +- 3 (26 to 35) identity,
> 33 +- 4 (27 to 39) spatial and 31 +- 6 (20 to 39) temporal, and had 17 +- 8 (4 to 25) incorrect
> trials per subject.

Correct trials kept are those of the derived epochs with accuracy 1 (`trialinfo` column 2, the
selection of the deposited scripts). The incorrect count is read (a) of all 120 trials, from the
per-subject accuracy of `docs/summary_file.csv` (over all 120 trials, by the data description;
40 trials per condition, Methods), and (b) of the trials kept.

In [ ]:
rows = []
for sid in fcx3.SUBJECTS:
    R = RES[sid]
    s = SUMMARY[SUMMARY.subject.str.lower() == sid].iloc[0]
    corr_all = int(round(40 * (s.acc_identity + s.acc_spatial + s.acc_temporal)))
    rows.append({"subject": sid, "trials kept": R["n_trials"], "correct kept": R["n_correct"],
                 "identity": R["n_correct_by_cond"][1], "spatial": R["n_correct_by_cond"][2], "temporal": R["n_correct_by_cond"][3],
                 "incorrect (a) of 120": 120 - corr_all, "incorrect (b) kept": R["n_incorrect"], "correct of 120": corr_all})
tc = pd.DataFrame(rows)
print(tc.to_string(index=False))
summ = tc.drop(columns="subject").agg(["mean", "std", "min", "max"]).round(2)
print("\n", summ.to_string())
exp = {"correct kept": (96, None), "identity": (32, (26, 35)), "spatial": (33, (27, 39)), "temporal": (31, (20, 39)),
       "incorrect (a) of 120": (17, (4, 25)), "incorrect (b) kept": (17, (4, 25))}
for k, (v, rg) in exp.items():
    m = tc[k].mean()
    within = abs(m - v) <= 0.1 * v
    rg_ok = None if rg is None else (tc[k].min() >= rg[0] - 0.1 * rg[0] and tc[k].max() <= rg[1] + 0.1 * rg[1])
    print(f"{k}: mean {m:.2f} vs {v} (within 10%: {within}); range {tc[k].min()}-{tc[k].max()} vs {rg} widened 10%: {rg_ok}")
print("\nCorrect trials kept equal all correct trials of 120 in:", list(tc.subject[tc["correct kept"] == tc["correct of 120"]]))

**Result: reproduced.** Correct trials kept average 95.71 per subject (paper 96); identity
32.14 +- 3.13 (26 to 35), spatial 32.86 +- 4.30 (27 to 39), temporal 30.71 +- 6.47 (20 to 39), the
paper's means and ranges. Incorrect trials kept (reading (b)) average 17.29 +- 8.44 (4 to 25),
the paper's 17 +- 8 (4 to 25); incorrect trials of all 120 (reading (a)) average 18.29 (6 to 29),
outside the widened range. In S1 to S5 every correct trial of the 120 was kept.

## fig1b-accuracy

> All subjects were proficient at the task (accuracy 0.63 to 0.98, chance 0.5), and accuracy did not
> differ between conditions.

Accuracy per subject and condition is in `docs/summary_file.csv`, over all 120 trials (data
description), which is reading (a) of B1's "all trials" question; the trial tables of S6 and S7
list only kept trials, so single-trial accuracy over all 120 is rebuilt as counts out of 40 per
condition (Methods: "120 trials divided evenly"). The logit mixed model (condition fixed, subject
random intercept) is fitted by maximum likelihood with Gauss-Hermite quadrature
(`fcx3.logit_mixed`; the paper names the model, not the software), with condition as one
numeric term (the single degree of freedom of the paper's F(1,19) for RT) and, as a variant, as
two dummy terms. Reading (b) of B1 (accuracy over kept trials) is shown as a variant.

In [ ]:
acc = SUMMARY[["subject", "acc_identity", "acc_spatial", "acc_temporal"]].copy()
acc["overall"] = acc[["acc_identity", "acc_spatial", "acc_temporal"]].mean(axis=1)
print(acc.round(3).to_string(index=False))
lo, hi = 0.63, 0.98
for name, vals in [("(a) per subject over conditions", acc.overall.values),
                   ("(b) per subject and condition", acc[["acc_identity", "acc_spatial", "acc_temporal"]].values.ravel())]:
    print(f"{name}: min {vals.min():.3f} (expected 0.63 +- 0.063: {abs(vals.min() - lo) <= 0.063}), max {vals.max():.3f} "
          f"(expected 0.98 +- 0.098: {abs(vals.max() - hi) <= 0.098}); all inside [{lo - 0.063:.3f}, {hi + 0.098:.3f}]: "
          f"{bool(((vals >= lo - 0.063) & (vals <= hi + 0.098)).all())}")

long = acc.melt(id_vars="subject", value_vars=["acc_identity", "acc_spatial", "acc_temporal"], var_name="cond", value_name="acc")
long["cond_code"] = long.cond.map({"acc_identity": 1, "acc_spatial": 2, "acc_temporal": 3})
long["k"] = np.round(long.acc * 40).astype(int)
long["n"] = 40
beta, se, ll1 = fcx3.logit_mixed(long.k.values, long.n.values, long.cond_code.values[:, None].astype(float), long.subject.values)
_, _, ll0 = fcx3.logit_mixed(long.k.values, long.n.values, np.zeros((len(long), 0)), long.subject.values)
z = beta[1] / se[1]
print(f"\n(a) all 120 trials, condition as one term: slope {beta[1]:.3f} (SE {se[1]:.3f}), Wald z {z:.2f}, p {2 * stats.norm.sf(abs(z)):.3f};"
      f" likelihood ratio chi2(1) {2 * (ll1 - ll0):.2f}, p {stats.chi2.sf(2 * (ll1 - ll0), 1):.3f}")
D = np.column_stack([(long.cond_code == 2).astype(float), (long.cond_code == 3).astype(float)])
_, _, ll2 = fcx3.logit_mixed(long.k.values, long.n.values, D, long.subject.values)
print(f"(a) condition as two dummy terms: likelihood ratio chi2(2) {2 * (ll2 - ll0):.2f}, p {stats.chi2.sf(2 * (ll2 - ll0), 2):.3f}")

kept = pd.concat([META[s]["trials"].assign(subject=s) for s in fcx3.SUBJECTS])
kk = kept.groupby(["subject", "condition"]).correct.agg(["sum", "count"]).reset_index()
b2, se2, l1 = fcx3.logit_mixed(kk["sum"].values, kk["count"].values, kk.condition.values[:, None].astype(float), kk.subject.values)
_, _, l0 = fcx3.logit_mixed(kk["sum"].values, kk["count"].values, np.zeros((len(kk), 0)), kk.subject.values)
print(f"(b) kept trials, condition as one term: slope {b2[1]:.3f}, likelihood ratio chi2(1) {2 * (l1 - l0):.2f}, p {stats.chi2.sf(2 * (l1 - l0), 1):.3f}")
print("accuracy over kept trials per subject:", kept.groupby("subject").correct.mean().round(3).to_dict())

**Result: reproduced.** Under reading (b), per subject and condition, accuracy runs from 0.625 to
0.975, the paper's 0.63 to 0.98; under reading (a), per subject, it runs from 0.758 to 0.950, and the
lowest value is outside 0.63 +- 0.063. The logit mixed model on all 120 trials finds no effect of
condition (likelihood ratio p = 0.283 with condition as one term, p = 0.160 as two terms; kept
trials p = 0.276), so the absence holds by the criterion (p > 0.05). The paper reports p > 0.50;
the p obtained here is lower than that.

## fig1b-rt-identity-slower

> Response times differed by condition, being longer on identity than on spatial and temporal
> trials, with no difference between spatial and temporal trials.

Reading (a) of B1: one mean correct-trial RT per subject and condition (`docs/summary_file.csv`,
"calculated on all correct trials"), a linear mixed model with condition as one numeric term and a
subject random intercept (REML, statsmodels), F = t^2 on the residual degrees of freedom (21 - 2 =
19; pairs 14 - 2 = 12), as fitlme reports. Reading (b): single correct trials; only the kept trials
are in the trial tables (all correct trials in S1 to S5), so (b) is on those. Cohen's d is given as
the mean paired difference over its SD across subjects, one common definition (the paper does not
say which it used; d is reported, not judged).

In [ ]:
import statsmodels.formula.api as smf

rt = SUMMARY.melt(id_vars="subject", value_vars=["rt_identity", "rt_spatial", "rt_temporal"], var_name="cond", value_name="rt")
rt["c"] = rt.cond.map({"rt_identity": 1, "rt_spatial": 2, "rt_temporal": 3})
print(SUMMARY[["subject", "rt_identity", "rt_spatial", "rt_temporal"]].round(0).to_string(index=False))


def lme_f(df, y, groups):
    r = smf.mixedlm(f"{y} ~ c", df, groups=df[groups]).fit(reml=True)
    t = r.tvalues["c"]
    return r.params["c"], t ** 2, stats.f.sf(t ** 2, 1, len(df) - 2), len(df) - 2


def report(df, y, groups, label):
    out = {}
    b, F, p, dfr = lme_f(df, y, groups)
    out["all"] = (b, F, p)
    print(f"{label} omnibus: slope {b:.1f} ms per condition step, F(1,{dfr}) = {F:.2f}, p = {p:.4f}")
    for a, bb, name in [(1, 2, "identity vs spatial"), (1, 3, "identity vs temporal"), (2, 3, "spatial vs temporal")]:
        sub = df[df.c.isin([a, bb])].copy()
        b2, F2, p2, dfr2 = lme_f(sub, y, groups)
        out[name] = (b2, F2, p2)
        print(f"   {name}: difference {b2 * (bb - a):.1f} ms ({'first longer' if b2 < 0 else 'second longer'}), F(1,{dfr2}) = {F2:.2f}, p = {p2:.4f}")
    return out


rt_a = report(rt, "rt", "subject", "(a) subject-condition means")
w = SUMMARY.set_index("subject")
for a, b in [("rt_identity", "rt_spatial"), ("rt_identity", "rt_temporal"), ("rt_spatial", "rt_temporal")]:
    d = w[a] - w[b]
    print(f"   d ({a[3:]} - {b[3:]}): {d.mean() / d.std(ddof=1):.2f}; identity longer in {int((d > 0).sum())}/7 subjects")
tr = kept[kept.correct == 1].rename(columns={"condition": "c"})
rt_b = report(tr, "rt", "subject", "(b) single kept correct trials")

**Result: reproduced.** On subject-condition means (reading (a)), condition affects RT
(F(1,19) = 6.04, p = 0.024), identity RT is longer than spatial (by 513 ms, F(1,12) = 11.81, p =
0.005) and temporal (by 392 ms, F(1,12) = 9.39, p = 0.010), and spatial and temporal do not differ
(p = 0.254). Identity RT is the longest in all 7 subjects. The F values are below the paper's
bounds (F > 6.50, > 13.78, > 10.95), and the d values (1.30, 1.16, by the definition used here)
below its 2.14 and 1.91. Single kept trials (reading (b)) give the same pattern (p = 0.018, 0.037,
0.004 and 0.844).

## erp-no-condition-difference

> ERPs over the processing interval did not differ between identity and spatial or temporal trials
> in any subject.

Main reading, per the claim's criterion and the Methods (Event-Related Potentials, paragraph 1):
the 200-ms pretrial baseline and 900-ms processing segments are cut, each zero padded to 7.5 s and
low-passed at 30 Hz (FIR), and the baseline is the pretrial temporal mean averaged over trials, as
erp_compute.m computes it (one value per channel, which cannot change a t-test). Variants: each
trial corrected on its own pretrial mean (the Methods' "post-cue processing - pretrial mean" allows
it); the Results' "1-30 Hz bandpass" in place of the low-pass; the whole derived epoch filtered
before the windows are cut (erp_compute.m); the main reading on the 900-ms-delay trials only (Data
check). Each is tested per subject by the cluster test of S2 over channels and time, identity
against spatial and against temporal, 1000 shuffles. Readings of the neighbours: adjacent,
distance, none; readings of the threshold: a cluster's p below 0.05 per tail (the Methods' "fewer
than 5% of randomizations yielded a larger effect", and FieldTrip's behaviour when cfg.alpha = 0.05
and the tails are not corrected), or below 0.025. The deposited scripts hold no cluster test, so
neither is settled by the code. The smallest cluster p per subject and contrast is printed for
each. With 1000 shuffles a cluster p near 0.04 has a Monte Carlo standard error of about 0.006.

In [ ]:
ERP_LABEL = {"methods": "segments padded, 30-Hz low-pass, baseline mean over trials (main)",
             "per_trial_baseline": "as main, per-trial baseline", "bandpass": "as main, 1-30 Hz band-pass",
             "code": "whole epoch filtered (erp_compute.m)", "methods_900": "as main, 900-ms-delay trials only"}
rows = []
for sid in fcx3.SUBJECTS:
    for config, byc in RES[sid]["erp"].items():
        for con, res in byc.items():
            for nb, v in res.items():
                pmin = min([p for _, p in v["pos"]] + [p for _, p in v["neg"]] + [1.0])
                rows.append({"config": config, "subject": sid, "contrast": con, "neighbours": nb, "min cluster p": pmin})
erp = pd.DataFrame(rows)
ERP_COUNTS = []
for config in fcx3.ERP_CONFIGS:
    print(f"==== {config}: {ERP_LABEL[config]}")
    e = erp[erp.config == config]
    print(e.pivot_table(index=["subject", "contrast"], columns="neighbours", values="min cluster p").round(3).to_string())
    for nb in fcx3.NB_READINGS:
        en = e[e.neighbours == nb]
        r = {"config": config, "neighbours": nb, "p < 0.05": int((en["min cluster p"] < 0.05).sum()),
             "p < 0.025": int((en["min cluster p"] < 0.025).sum()), "p <= 0.10": int((en["min cluster p"] <= 0.10).sum()),
             "smallest p": round(en["min cluster p"].min(), 3)}
        ERP_COUNTS.append(r)
ERP_COUNTS = pd.DataFrame(ERP_COUNTS)
print("\nSubject-contrasts (of 14) with a cluster at each threshold (paper: none, all p > 0.10):")
print(ERP_COUNTS.to_string(index=False))
print("\nCorrect trials with a 900-ms delay, per condition:", {sid: RES[sid]["n_correct_900_by_cond"] for sid in fcx3.SUBJECTS})

**Result: not reproduced, resting on an assumption.** Under the main reading with neighbours
adjacent by contact number and clusters judged at 0.05 per tail, S2 has a significant cluster for
identity against temporal (p = 0.041); no other subject-contrast is below 0.05, and 2 of 14 have
p <= 0.10 (the paper reports all p > 0.10). The outcome depends on the readings: at 0.025 per tail
the adjacent reading finds no significant cluster (reproduced), while the distance reading (S5,
p = 0.014) and the reading without spatial clustering (S2, p = 0.024) find one at either
threshold. Among the variants, the 1-30 Hz band-pass gives no cluster below 0.05 with adjacent
neighbours (smallest p 0.085); the per-trial baseline gives one (S5 identity against spatial,
p = 0.030); filtering the whole epoch gives S2 p = 0.031; the 900-ms-delay trials alone give one
(S5 identity against temporal, p = 0.023). The deciding p values lie between 0.02 and 0.05, so
the outcome is close to the threshold under every reading.

## fig2-spatial-power and fig3-temporal-power

> In the identity against spatial contrast of processing-interval power, 5 of 7 subjects showed
> spatial > identity effects (54 +- 22% of electrodes per subject) and 2 of 7 showed identity >
> spatial effects (36 +- 12%).

> In the identity against temporal contrast of processing-interval power, 4 of 7 subjects showed
> temporal > identity effects (68 +- 30% of electrodes per subject) and 2 of 7 showed identity >
> temporal effects (37 +- 10%).

Power with 3-cycle Hanning windows, 2 to 40 Hz, 10-ms steps over the 900-ms processing segment.
Main reading: the segment alone, embedded in zeros, every window computed (Methods, Spectral
Decomposition, "Data segments were zero-padded to 7,500 ms"; S1 reading (a)). Variant: the windows
computed on the whole derived epoch and the processing window cut afterwards, as power_compute.m
does (its lines are printed in the Data check), so that windows longer than the segment (below
about 3.3 Hz) take in the cue and the response period. A further variant uses the main reading on
the 900-ms-delay trials only. Power is z-scored on the bootstrapped pretrial baseline of all correct
trials (zbaseline.m; r = all correct trials, settling S1's open r), which cannot change the
t-tests. The cluster test of S2 over channel, frequency and time, 1000 shuffles. An electrode shows
an effect of a sign when any of its points lies in a significant cluster of that sign (S2 reading
(a)); reading (b) assigns an electrode in both to the sign with the larger summed t. Every
combination of spectra, neighbour reading, threshold and assignment is shown, and the per-subject
percentages under each reading.

In [ ]:
def power_effects(contrast, nb, alpha=0.05, assign="a", config="methods"):
    out = []
    for sid in fcx3.SUBJECTS:
        cl = RES[sid]["power"][config][contrast]["clusters"][nb]
        n = len(RES[sid]["labels"])
        ident, other = np.zeros(n, bool), np.zeros(n, bool)
        ts_i, ts_o = np.zeros(n), np.zeros(n)
        for c in cl["pos"]:  # positive t: identity > other
            if c["p"] < alpha:
                ident |= c["elec_mask"]
                ts_i += c["tsum_by_elec"]
        for c in cl["neg"]:
            if c["p"] < alpha:
                other |= c["elec_mask"]
                ts_o += -c["tsum_by_elec"]
        if assign == "b":
            both = ident & other
            ident = ident & ~(both & (ts_o >= ts_i))
            other = other & ~(both & (ts_i > ts_o))
        out.append({"subject": sid, "n": n, "other>identity": int(other.sum()), "identity>other": int(ident.sum()),
                    "p other>identity": cl["neg_minp"], "p identity>other": cl["pos_minp"]})
    return pd.DataFrame(out)


POWER_LABEL = {"methods": "segment zero-padded (main)", "code": "whole-epoch windows (power_compute.m)",
               "methods_900": "segment zero-padded, 900-ms-delay trials only"}
EXPECTED = {"spatial": (5, 2, 54, 36), "temporal": (4, 2, 68, 37)}
POWER_SUMMARY = []
PER_SUBJECT = {}
for config in fcx3.POWER_CONFIGS:
    for contrast in ["spatial", "temporal"]:
        for nb in fcx3.NB_READINGS:
            for alpha in (0.05, 0.025):
                for assign in ("a", "b"):
                    e = power_effects(contrast, nb, alpha, assign, config)
                    po = 100 * e["other>identity"] / e.n
                    pi = 100 * e["identity>other"] / e.n
                    PER_SUBJECT[(config, contrast, nb, alpha, assign)] = [f"{a:.0f}/{b:.0f}" for a, b in zip(po, pi)]
                    so, si = po[po > 0], pi[pi > 0]
                    ex = EXPECTED[contrast]
                    row = {"config": config, "contrast": contrast, "neighbours": nb, "alpha per tail": alpha, "assign": assign,
                           "subjects other>id": len(so), "subjects id>other": len(si),
                           "% other>id": f"{so.mean():.1f} +- {so.std(ddof=1):.1f}" if len(so) > 1 else (f"{so.mean():.1f}" if len(so) else "-"),
                           "% id>other": f"{si.mean():.1f} +- {si.std(ddof=1):.1f}" if len(si) > 1 else (f"{si.mean():.1f}" if len(si) else "-"),
                           "subjects other>id (list)": list(e.subject[po > 0]), "subjects id>other (list)": list(e.subject[pi > 0])}
                    row["counts match"] = len(so) == ex[0] and len(si) == ex[1]
                    row["% match"] = (len(so) > 0 and len(si) > 0 and abs(so.mean() - ex[2]) <= 0.1 * ex[2]
                                      and abs(si.mean() - ex[3]) <= 0.1 * ex[3])
                    row["counts within 1"] = abs(len(so) - ex[0]) <= 1 and abs(len(si) - ex[1]) <= 1
                    POWER_SUMMARY.append(row)
POWER_SUMMARY = pd.DataFrame(POWER_SUMMARY)
for config in fcx3.POWER_CONFIGS:
    print(f"==== {config}: {POWER_LABEL[config]}")
    print(POWER_SUMMARY[POWER_SUMMARY.config == config].drop(columns=["config", "subjects other>id (list)", "subjects id>other (list)"]).to_string(index=False))
print("\nSubjects per effect, main spectra, 0.05 per tail, assignment (a):")
print(POWER_SUMMARY[(POWER_SUMMARY.config == "methods") & (POWER_SUMMARY["alpha per tail"] == 0.05) & (POWER_SUMMARY["assign"] == "a")][
    ["contrast", "neighbours", "subjects other>id (list)", "subjects id>other (list)"]].to_string(index=False))

In [ ]:
print("Per-subject % of electrodes, 'other > identity / identity > other', main spectra (paper: per-subject values only in Figures 2A and 3A, not read)")
for contrast in ["spatial", "temporal"]:
    tab = pd.DataFrame({f"{nb[:4]} {alpha} {assign}": PER_SUBJECT[("methods", contrast, nb, alpha, assign)]
                        for nb in fcx3.NB_READINGS for alpha in (0.05, 0.025) for assign in ("a", "b")}, index=fcx3.SUBJECTS)
    print(f"identity against {contrast} (columns: neighbours, alpha per tail, assignment)")
    print(tab.to_string())
print("\nSmallest cluster p of each sign and subject, main spectra; paper: all p < 0.04 for the effects it reports")
rows = []
for contrast in ["spatial", "temporal"]:
    for sid in fcx3.SUBJECTS:
        cl = RES[sid]["power"]["methods"][contrast]["clusters"]
        rows.append({"contrast": contrast, "subject": sid, **{f"{nb} other>id": round(cl[nb]["neg_minp"], 3) for nb in fcx3.NB_READINGS},
                     **{f"{nb} id>other": round(cl[nb]["pos_minp"], 3) for nb in fcx3.NB_READINGS}})
print(pd.DataFrame(rows).to_string(index=False))

**Result: partially reproduced.** Under the main spectra, 5 subjects show spatial > identity
effects (S2 to S6) under every neighbour reading and threshold, and 1 shows identity > spatial
effects (S4; paper 2) under every reading but adjacent neighbours at 0.025 per tail, which finds none. With adjacent neighbours at 0.05 per tail the percentages
are 37.7 +- 21.1% and 11.8% (S4 alone), below the paper's 54 +- 22% and 36 +- 12%. The counts
match within one subject and 5 of 7 subjects have a significant cluster, the criterion's partial
match. The paper's counts (5 and 2) appear only with the whole-epoch windows of power_compute.m,
where S7 adds an identity > spatial effect; there the distance reading gives 52.0% (within
tolerance of 54) and 28.4% (outside tolerance of 36), so no reading matches both percentages. The
significant clusters of the main spectra have p <= 0.028. On the 900-ms-delay trials alone 2 or 3
subjects show spatial > identity effects and none identity > spatial.

**Result: partially reproduced.** Under the main spectra, adjacent neighbours, 0.05 per tail and
assignment (a), 5 subjects show temporal > identity effects (paper 4) and 2 identity > temporal
effects (S1, S7; paper 2), with 37.4 +- 20.5% and 21.0 +- 17.4% of electrodes (paper 68 +- 30% and
37 +- 10%). Assignment (b) gives the paper's counts, 4 and 2, with 41.2% and 21.0%. No reading of
spectra, neighbours, threshold or assignment brings both percentages within tolerance; the
distance reading gives 47.8 to 55.2% for temporal > identity. The significant clusters of the
reported effects have p <= 0.014 with adjacent neighbours, except S2, whose temporal > identity
cluster is significant only under the distance reading (p = 0.020) and without spatial clustering
(p = 0.044). On the 900-ms-delay trials alone 1 to 3 subjects show temporal > identity effects.

## power-difficulty-subjects

> The subjects with identity > spatial or temporal power effects were the same subjects who had both
> longer RTs and lower accuracy on identity trials.

Subjects with an identity > spatial or identity > temporal effect are taken from the power claims
(every spectra, neighbour and threshold reading, assignment (a)); accuracy (all 120 trials) and
mean correct RT per condition from `docs/summary_file.csv`.

In [ ]:
w = SUMMARY.copy()
w["subject"] = w.subject.str.lower()
w = w.set_index("subject")
beh = pd.DataFrame({
    "RT id > sp": w.rt_identity > w.rt_spatial, "acc id < sp": w.acc_identity < w.acc_spatial,
    "RT id > tp": w.rt_identity > w.rt_temporal, "acc id < tp": w.acc_identity < w.acc_temporal,
    "RT id > mean(sp,tp)": w.rt_identity > (w.rt_spatial + w.rt_temporal) / 2,
    "acc id < mean(sp,tp)": w.acc_identity < (w.acc_spatial + w.acc_temporal) / 2})
beh["both (a) spatial"] = beh["RT id > sp"] & beh["acc id < sp"]
beh["both (a) temporal"] = beh["RT id > tp"] & beh["acc id < tp"]
beh["both (b)"] = beh["RT id > mean(sp,tp)"] & beh["acc id < mean(sp,tp)"]
print(beh.to_string())
for config in fcx3.POWER_CONFIGS:
    for nb in fcx3.NB_READINGS:
        for alpha in (0.05, 0.025):
            sel = POWER_SUMMARY[(POWER_SUMMARY.config == config) & (POWER_SUMMARY.neighbours == nb) & (POWER_SUMMARY["alpha per tail"] == alpha)
                                & (POWER_SUMMARY["assign"] == "a")].set_index("contrast")
            sp_ = set(sel.loc["spatial", "subjects id>other (list)"])
            tp_ = set(sel.loc["temporal", "subjects id>other (list)"])
            effect = sorted(sp_ | tp_)
            ok_a = all((s not in sp_ or beh.loc[s, "both (a) spatial"]) and (s not in tp_ or beh.loc[s, "both (a) temporal"]) for s in effect)
            ok_b = all(beh.loc[s, "both (b)"] for s in effect)
            others_a = [s for s in beh.index if s not in effect and (beh.loc[s, "both (a) spatial"] or beh.loc[s, "both (a) temporal"])]
            others_b = [s for s in beh.index if s not in effect and beh.loc[s, "both (b)"]]
            print(f"{config}, {nb}, {alpha}: identity>spatial {sorted(sp_)}, identity>temporal {sorted(tp_)}; every one has both under (a): {ok_a},"
                  f" under (b): {ok_b}; other subjects with both, (a): {others_a}, (b): {others_b}")

**Result: reproduced.** Under the main spectra the subjects with identity > spatial or temporal
effects are S1, S4 and S7 under every neighbour reading at 0.05 per tail (S4 for spatial, S1 and S7
for temporal, and S4 also for temporal without spatial clustering). Under reading (b), identity
against the mean of spatial and temporal, each has longer identity RT and lower identity accuracy;
under reading (a) S1 fails for the temporal contrast, where its identity and temporal accuracy are
equal (0.825). The same holds under every other configuration printed. S3 also has both without an
identity > effect, so the converse ("the same subjects", reading (ii)) does not hold. Identity RT is
the longest in all 7 subjects, so only accuracy separates subjects here.

## power-low-frequencies

> Significant processing-interval power effects were most prominent at low frequencies, in distinct
> delta-theta (2 to 7 Hz) and alpha-beta (9 to 24 Hz) ranges, whatever the contrast.

Per subject and effect, the t values inside significant clusters (0.05 per tail) are averaged over
electrodes and time per frequency, (a) over the masked points only, (b) over all points with the
unmasked as zero; then averaged, in absolute value, over the subjects with the effect. The rule
(2-24 Hz larger than 25-40 Hz, local maxima in 2-7 and 9-24 Hz) is the claims file's. A point at
either end of the 2-40 Hz range counts as a local maximum when it exceeds its one neighbour (the
usual definition on a closed range); the result without counting the 2-Hz end is shown as well.
Every neighbour reading is shown, for the main spectra and the whole-epoch variant.

In [ ]:
def density(nb, effect, reading, alpha=0.05, config="methods"):
    out = {}
    for sid in fcx3.SUBJECTS:
        R = RES[sid]
        dens = []
        for contrast in (["spatial"] if effect == "spatial>identity" else ["temporal"] if effect == "temporal>identity" else ["spatial", "temporal"]):
            cl = R["power"][config][contrast]["clusters"][nb]
            sign = "neg" if effect != "identity>other" else "pos"
            cs = [c for c in cl[sign] if c["p"] < alpha]
            if not cs:
                continue
            tm = sum(c["t_masked"].astype(float) for c in cs)  # ch x f x t
            if reading == "a":
                cnt = (tm != 0).sum(axis=(0, 2))
                dens.append(np.where(cnt > 0, tm.sum(axis=(0, 2)) / np.maximum(cnt, 1), 0))
            else:
                dens.append(tm.mean(axis=(0, 2)))
        if dens:
            out[sid] = np.abs(np.mean(dens, axis=0))
    return out


f = fcx3.FOI
LF = []
for config in ("methods", "code"):
    for nb in fcx3.NB_READINGS:
        for reading in ("a", "b"):
            for effect in ["spatial>identity", "temporal>identity", "identity>other"]:
                d = density(nb, effect, reading, config=config)
                if not d:
                    LF.append({"config": config, "neighbours": nb, "reading": reading, "effect": effect, "subjects": 0})
                    continue
                m = np.mean(list(d.values()), axis=0)
                low, high = m[(f >= 2) & (f <= 24)].mean(), m[(f >= 25) & (f <= 40)].mean()
                peaks = [int(f[i]) for i in range(len(f)) if (i == 0 or m[i] > m[i - 1]) and (i == len(f) - 1 or m[i] >= m[i + 1])]
                inner = [p for p in peaks if 2 < p < 40]
                LF.append({"config": config, "neighbours": nb, "reading": reading, "effect": effect, "subjects": len(d), "2-24 Hz": round(low, 3),
                           "25-40 Hz": round(high, 3), "low > high": low > high, "local maxima (Hz)": peaks,
                           "max in 2-7": any(2 <= p <= 7 for p in peaks), "max in 9-24": any(9 <= p <= 24 for p in peaks),
                           "max in 2-7, 2-Hz edge not counted": any(2 < p <= 7 for p in inner), "peak Hz": int(f[np.argmax(m)])})
LF = pd.DataFrame(LF)
print(LF.to_string(index=False))
ok = LF[LF.subjects > 0].groupby(["config", "neighbours", "reading"]).apply(
    lambda g: pd.Series({"all three meet the rule": bool((g["low > high"] & g["max in 2-7"] & g["max in 9-24"]).all() and len(g) == 3),
                         "same, 2-Hz edge not counted": bool((g["low > high"] & g["max in 2-7, 2-Hz edge not counted"] & g["max in 9-24"]).all() and len(g) == 3)}))
print("\nCombinations where every effect meets the claims file's rule:")
print(ok.to_string())
fig, axes = plt.subplots(1, 3, figsize=(12, 3), sharey=False)
for ax, effect in zip(axes, ["spatial>identity", "temporal>identity", "identity>other"]):
    for sid, v in density("adjacent", effect, "b").items():
        ax.plot(f, v, lw=0.8, label=sid)
    ax.set_title(effect + " (main spectra, adjacent, b)")
    ax.set_xlabel("Hz")
axes[0].set_ylabel("|mean significant t|")
axes[0].legend(fontsize=7)
plt.tight_layout()
plt.show()
print("All densities plotted are within the axes (autoscaled).")

**Result: reproduced, resting on an assumption.** Under the main spectra with the density over all
points (reading (b)), every neighbour reading meets the rule. With adjacent neighbours, each effect
is larger over 2 to 24 Hz than over 25 to 40 Hz (0.099 against 0.021 for spatial > identity, 0.097
against 0.032 for temporal > identity, 0.057 against 0.018 for identity > other), with local maxima
in 2 to 7 Hz (2, 3 and 2 Hz) and in 9 to 24 Hz. The match rests on counting the 2-Hz end of the range
as a local maximum: without it, the identity > other density (maxima at 2 and 14 Hz) has no maximum
in 2 to 7 Hz in any combination with reading (b), which would be the partial match. Over masked points only
(reading (a)) the identity > other density is larger above 24 Hz under every combination. With the
whole-epoch windows only the distance reading meets the rule.

## Task-induced PSI: fig4-maintenance-parietal-led, fig4a-delta-theta-shift, fig4c-alpha-beta-shift, directional-pairs

> During maintenance, PFC-parietal connectivity was unidirectional, parietal to PFC, in both the
> delta-theta (z < -40.03 +- 8.73) and alpha-beta (z < -30.37 +- 2.60) ranges.

> After the test cue, delta-theta directionality shifted from a unidirectional parietal-led network
> to a bidirectional frontoparietal one, with net PFC-led connectivity above zero in 6 of 7 subjects
> during processing (z > 6.01 +- 6.42).

> Alpha-beta directionality also shifted with the test cue, but all subjects remained net
> parietal-led during processing (z < -12.60 +- 7.51).

> During processing, 1008 pairs were significantly directional in the delta-theta range and 984 in
> the alpha-beta range.

PSI per P1, with the details psi_compute.m and psianalysis.m settle: the ERP subtracted is the mean
over all correct trials (P1 "raw trial-wise mean", reading (a); subtracterp.m with one condition);
spectra are one Hanning-tapered FFT of each 900-ms segment padded to 7.5 s ('mtmfft', P1 spectra
reading (a)); PSI is one value per band summed over adjacent 1-Hz bins (band reading (a)). PSI is
computed (a) from the trial-averaged coherency (Nolte et al. 2008) or (b) per trial from the
per-trial normalised cross-spectrum, averaged over trials (the structure of psianalysis.m), with
the conjugate the Methods' cross-spectral density implies.

The null follows the Methods: "we randomly shuffled the frequencies in one signal and re-computed
PSI 1,000 times". Each shuffle permutes the frequency bins of the parietal spectrum (the same
permutation for every trial and pair, P1 null reading (a)) and recomputes the same statistic, so
that the null and the observed value measure the same thing (`fcx3.psi_z`). psianalysis.m builds
its null differently, from the slope of the trial-mean spectrum with both factors shuffled, which
is not the statistic it z-scores; reading (b) against that null is shown as a variant (`b_code`).
The deposited code without the conjugate, whose measure equals plus or minus exp(2i x the second
channel's phase) and so depends on one channel only, is shown with either channel first for the
record of that code mismatch; it is not a reading of the paper and decides no outcome. As a check of
calibration on these data, each subject's processing PSI is also computed with the parietal trials
paired with other trials' PFC data, which removes any trial-locked coupling; a calibrated z then
exceeds 1.96 in about 5% of pairs. Positive z: PFC leads.

In [ ]:
READ = {"a": "(a) trial-averaged coherency", "b": "(b) per-trial PSI averaged",
        "b_code": "(b) against psianalysis.m's null (variant)",
        "code": "deposited code, PFC first (not a reading)", "code_rev": "deposited code, parietal first, sign reversed (not a reading)"}
psi_rows = []
for sid in fcx3.SUBJECTS:
    for band in fcx3.BANDS:
        for iv in ("maintenance", "processing"):
            zs = {**RES[sid]["psi"][band][iv], **RES[sid]["psi_deposited"][band][iv]}
            for k, z in zs.items():
                for (pfc, par, hemi), v in zip(RES[sid]["pairs"], z):
                    psi_rows.append({"subject": sid, "band": band, "interval": iv, "reading": k, "pair": f"{sid}:{pfc}-{par}", "z": v})
PSI = pd.DataFrame(psi_rows)
print("Pairs per subject:", PSI[(PSI.band == "delta-theta") & (PSI.interval == "maintenance") & (PSI.reading == "a")].groupby("subject").size().to_dict())
print("Frequencies read (Hz):", {b: np.round(RES[fcx3.SUBJECTS[0]]['psi'][b]['freqs'], 3).tolist() for b in fcx3.BANDS})
persub = PSI.groupby(["band", "interval", "reading", "subject"]).z.mean().unstack("subject")
print("\nPer-subject mean PSI z over pairs:")
print(persub.round(2).to_string())
grp = PSI.groupby(["band", "interval", "reading", "subject"]).z.mean().groupby(["band", "interval", "reading"]).agg(["mean", "std", "min", "max"])
grp.columns = ["mean over subjects", "SD over subjects", "min subject", "max subject"]
pairs_ = PSI.groupby(["band", "interval", "reading"]).z.agg(["mean", "std", "min", "max", lambda z: np.mean(np.abs(z) > 1.96)])
pairs_.columns = ["mean over pairs", "SD over pairs", "min pair", "max pair", "fraction |z| > 1.96"]
print("\n", pd.concat([grp, pairs_], axis=1).round(2).to_string())

In [ ]:
print("Calibration on the data: processing PSI with each PFC trial paired with another trial's parietal data")
print("(trial order shuffled within subject, so no trial-locked coupling; a calibrated z gives |z| > 1.96 in about 5% of pairs)")
rows = []
for band in fcx3.BANDS:
    for k in ("a", "b", "b_code"):
        z = np.concatenate([RES[sid]["psi_surrogate"][band][k] for sid in fcx3.SUBJECTS])
        by = {sid: round(float(np.mean(np.abs(RES[sid]["psi_surrogate"][band][k]) > 1.96)), 2) for sid in fcx3.SUBJECTS}
        rows.append({"band": band, "reading": READ[k], "pairs": len(z), "fraction |z| > 1.96": round(float(np.mean(np.abs(z) > 1.96)), 3),
                     "SD of z": round(float(z.std()), 2), "by subject": by})
print(pd.DataFrame(rows).to_string(index=False))
print("Trials left in place by the shuffle:", {sid: RES[sid]["psi_surrogate"]["delta-theta"]["same_trial_fraction"] for sid in fcx3.SUBJECTS})

In [ ]:
print("fig4-maintenance-parietal-led: expected delta-theta -40.03 (tol 4.0 or more negative), alpha-beta -30.37 (tol 3.04 or more negative)")
for k in ("a", "b", "b_code", "code", "code_rev"):
    m = persub.xs(("maintenance", k), level=("interval", "reading"))
    for band, exp, tol in [("delta-theta", -40.03, 4.0), ("alpha-beta", -30.37, 3.04)]:
        v = m.loc[band]
        pooled = PSI[(PSI.band == band) & (PSI.interval == "maintenance") & (PSI.reading == k)].z
        print(f"  {READ[k]}, {band}: subjects below -1.96: {int((v < -1.96).sum())}/7; mean over subjects {v.mean():.2f} +- {v.std(ddof=1):.2f};"
              f" pooled pairs {pooled.mean():.2f} +- {pooled.std(ddof=1):.2f}; pairs below -1.96 {int((pooled < -1.96).sum())}/{len(pooled)},"
              f" above 1.96 {int((pooled > 1.96).sum())}; within tolerance or beyond: (a) {v.mean() <= exp + tol}, (b) {pooled.mean() <= exp + tol}")

**Result: not reproduced.** Under neither PSI reading is maintenance parietal-led in any subject:
subject means are -0.39 to 0.15 (delta-theta) and -0.67 to 0.04 (alpha-beta) under reading (a), and
-0.67 to 0.05 and -0.48 to -0.05 under reading (b), none below -1.96; the paper reports -40.03 +-
8.73 and -30.37 +- 2.60. Against psianalysis.m's null, reading (b) puts 4 of 7 subjects below -1.96
in delta-theta (mean -0.28 +- 10.69) and none in alpha-beta, but that null gives |z| > 1.96 in 90%
and 92% of trial-shuffled pairs, so it does not measure direction. The deposited code without the
conjugate gives every subject below -1.96 with the parietal channel first (means -55.47 and -69.83)
and above +1.96 with the PFC channel first (+58.67 and +69.48); its sign follows the channel order.

In [ ]:
LME = {}
for band in fcx3.BANDS:
    for k in ("a", "b", "b_code", "code", "code_rev"):
        d = PSI[(PSI.band == band) & (PSI.reading == k)].pivot_table(index=["subject", "pair"], columns="interval", values="z").reset_index()
        eff, F, p, n = fcx3.paired_lme(d.maintenance.values, d.processing.values, d.subject.values)
        LME[(band, k)] = (eff, F, p, n)
        line = f"{band}, {READ[k]}: processing - maintenance {eff:.2f}, F(1,{2 * n - 2}) = {F:.1f}, p = {p:.3g} (pairs {n})"
        if k in ("a", "b"):
            e2, F2, p2 = fcx3.mixedlm_check(d.maintenance.values, d.processing.values, d.subject.values, d.pair.values)
            line += f"; statsmodels MixedLM: {e2:.2f}, F {F2:.1f}, p {p2:.3g}"
            # pairs share electrodes; a subject-level test does not treat them as independent
            ps = d.groupby("subject")[["maintenance", "processing"]].mean()
            tt = stats.ttest_rel(ps.processing, ps.maintenance)
            line += f"; subject means, paired t(6) = {tt.statistic:.2f}, p {tt.pvalue:.3g}"
        print(line)
print("Paper: delta-theta F(1,2256) = 1936.10, p < 8e-306; alpha-beta F(1,2256) = 2302.30, p = 0 (processing > maintenance).")

In [ ]:
print("fig4a-delta-theta-shift: expected 6 subjects with processing z > 0, mean 6.01 (tol 0.6 or larger), F 1936.10 +- 193.6")
for k in ("a", "b", "b_code", "code", "code_rev"):
    v = persub.loc[("delta-theta", "processing", k)]
    pos = v[v > 0]
    eff, F, p, n = LME[("delta-theta", k)]
    print(f"  {READ[k]}: subjects above 0: {int((v > 0).sum())}/7 {list(v[v > 0].index)}; mean of 7 {v.mean():.2f} +- {v.std(ddof=1):.2f};"
          f" mean of those above 0 {pos.mean() if len(pos) else float('nan'):.2f}; model effect {eff:+.2f}, F {F:.1f}, p {p:.3g}")
print("fig4c-alpha-beta-shift: expected every subject below 0 in processing, mean -12.60 (tol 1.26 or more negative), F 2302.30 +- 230.2")
for k in ("a", "b", "b_code", "code", "code_rev"):
    v = persub.loc[("alpha-beta", "processing", k)]
    eff, F, p, n = LME[("alpha-beta", k)]
    print(f"  {READ[k]}: subjects below 0: {int((v < 0).sum())}/7; mean {v.mean():.2f} +- {v.std(ddof=1):.2f}; model effect {eff:+.2f}, F {F:.1f}, p {p:.3g}")

**Result: not reproduced.** No subject has net delta-theta PSI z above 0 in processing under
either reading (means -0.17 +- 0.16 under (a), -0.43 +- 0.34 under (b); paper 6.01 +- 6.42 with 6
subjects above 0). Processing exceeds maintenance under reading (b) in the pair-level model (by 0.27,
F(1,2268) = 114.2, p = 4.9e-26; paper F = 1936.10) but not under reading (a) (0.06, p = 0.14), and
on subject means the difference is absent under both (paired t(6) p = 0.43 and 0.95). The criterion's
partial match needs at least 4 subjects above 0. Against psianalysis.m's null, reading (b) gives 4
subjects above 0 (mean of those 6.22) and F = 59.5, the round-1 result, but that null is not
calibrated (trial-shuffled pairs pass |z| > 1.96 in 90% of cases).

**Result: partially reproduced, resting on an assumption.** Under PSI reading (b), adopted (see
the Summary), 6 of 7 subjects have processing alpha-beta PSI z below 0 (S3 is above, 0.46) and
processing exceeds maintenance in the pair-level model (by 0.43, F(1,2268) = 97.9, p = 1.2e-22;
paper F = 2302.30), the criterion's partial match. The effect is small: the subject means lie
between -1.35 and 0.46 (mean -0.57 +- 0.59; paper -12.60 +- 7.51), and on subject means the
interval difference reverses sign and is not significant (paired t(6) = -1.22, p = 0.27), since S3
holds 756 of the 1135 pairs. Under reading (a) 5 subjects are below 0 (mean -0.14), with processing
higher by 0.11 (p = 0.0097), which would not be reproduced.

In [ ]:
print("directional-pairs: pairs with processing |z| > 1.96, expected delta-theta 1008 (+-100.8), alpha-beta 984 (+-98.4), of all pairs")
SEL = {}
for k in ("a", "b", "b_code", "code", "code_rev"):
    for band, exp in [("delta-theta", 1008), ("alpha-beta", 984)]:
        p = PSI[(PSI.band == band) & (PSI.interval == "processing") & (PSI.reading == k)]
        n = int((p.z.abs() > 1.96).sum())
        SEL[(band, k)] = set(p.pair[p.z.abs() > 1.96])
        sur = ""
        if k in ("a", "b", "b_code"):
            zs = np.concatenate([RES[sid]["psi_surrogate"][band][k] for sid in fcx3.SUBJECTS])
            sur = f"; trial-shuffled surrogate {int((np.abs(zs) > 1.96).sum())}"
        print(f"  {READ[k]}, {band}: {n} of {len(p)} ({n / len(p):.1%}); within tolerance {abs(n - exp) <= 0.1 * exp}{sur};"
              f" by subject {p[p.z.abs() > 1.96].groupby('subject').size().to_dict()}")
print("Expected under a calibrated null with |z| > 1.96 at 5%:", round(0.05 * len(p)))

**Result: not reproduced.** With the Methods' null, 78 (reading (a)) or 1 (reading (b)) of 1135
pairs have processing |z| > 1.96 in delta-theta, and 104 or 74 in alpha-beta, against the paper's
1008 and 984. The trial-shuffled surrogate gives 68 and 78 under (a) and 1 and 71 under (b), and a
null at 5% would give about 57, so the counts are near what independent signals give. Against
psianalysis.m's null, reading (b) gives 1014 and 1042, within tolerance, but the trial-shuffled
surrogate gives 1022 and 1045 under the same null, so that count does not reflect directionality.

## Time-resolved PSI: fig5a-delta-theta-epochs, fig5a-identity-direction, fig5a-spatiotemporal-bidirectional, fig5b-alpha-beta-no-sustained

> Time-resolved delta-theta PSI differed between conditions in sustained epochs at 0 to 120 ms and
> 470 to 610 ms (identity against spatial) and at 0 to 170 ms and 660 to 860 ms (identity against
> temporal) after the test cue, the spatial late epoch ending about 50 ms before the temporal one
> begins.

> In identity trials the PFC led parietal sites early in processing (z > 2.62 and z > 2.35 in the
> two contrasts' early epochs) and the parietal cortex led later (z < -3.04 and z < -3.57).

> Delta-theta PSI during the selection of spatial and of temporal information stayed bidirectional
> (net near zero) throughout the processing interval.

> Time-resolved alpha-beta PSI showed significant condition differences at some time points but no
> sustained effect for either condition in either contrast.

Per P2 and psi_compute.m: per condition, the condition's ERP removed (subtracterp.m with three
conditions), 3-cycle Hanning windows at 10-ms steps ('mtmconvol'), PSI z per pair and time point
against its own null (the Methods' frequency shuffle of one signal, as for the task-induced PSI),
on the pairs with processing |z| > 1.96 under the same reading (the selection of
directional-pairs). Main spectra: the 900-ms segment zero padded (Methods, Time-Resolved PSI,
"Spectral decomposition ... otherwise identical"); variant: windows on the whole derived epoch, as
psi_compute.m computes them. subtracterp.m selects the temporal trials with `trialinfo(:,5)==2`,
the delay column, which selects none; the condition column (4) is used here. At each time point the
mixed model (condition fixed, subject and pair random) is computed as the paired test over pairs
(`fcx3.paired_lme`, checked against statsmodels below), FDR (Benjamini-Hochberg) over the 91 time
points at 0.05, sustained when a run of significant points of one sign spans at least 100 ms (the
runs longer than 100 ms are printed as well, for the Figure 5 legend's ">100 ms").

In [ ]:
TIMES = np.round(RES[fcx3.SUBJECTS[0]]["power_toi"] * 1000).astype(int)
TCFG = {"methods": "segment zero-padded (main)", "code": "whole-epoch windows (psi_compute.m)"}
TR = {}
for config in fcx3.PSI_T_CONFIGS:
    for band in fcx3.BANDS:
        for k in ("a", "b"):
            z = {c: [] for c in (1, 2, 3)}
            subj, pair = [], []
            for sid in fcx3.SUBJECTS:
                names = [f"{sid}:{a}-{b}" for a, b, _ in RES[sid]["pairs"]]
                keep = np.array([nm in SEL[(band, k)] for nm in names])
                for c in (1, 2, 3):
                    z[c].append(RES[sid]["psi_t"][config][band][c][k][keep])
                subj += [sid] * int(keep.sum())
                pair += [nm for nm, kk in zip(names, keep) if kk]
            TR[(config, band, k)] = {"z": {c: np.concatenate(v) for c, v in z.items()}, "subject": np.array(subj), "pair": np.array(pair)}


def contrast_stats(config, band, k, other):
    d = TR[(config, band, k)]
    p, eff = [], []
    for ti in range(len(TIMES)):
        e, F, pv, n = fcx3.paired_lme(d["z"][other][:, ti], d["z"][1][:, ti], d["subject"])
        p.append(pv)
        eff.append(e)  # identity minus other
    p, eff = np.array(p), np.array(eff)
    sig = fcx3.fdr_bh(p) if len(d["subject"]) > 1 else np.zeros(len(TIMES), bool)
    runs = []
    for sign in (1, -1):
        for a, b in fcx3.runs(sig & (np.sign(eff) == sign), TIMES):
            runs.append((int(a), int(b), sign, int(b - a) >= 100))
    return p, eff, sig, sorted(runs)


def grand(config, band, k, c):
    d = TR[(config, band, k)]
    per = pd.DataFrame(d["z"][c]).groupby(d["subject"]).mean()
    return per.mean(axis=0).values, per.std(axis=0, ddof=1).values / np.sqrt(len(per)), per


STATS = {}
for config in fcx3.PSI_T_CONFIGS:
    for band in fcx3.BANDS:
        for k in ("a", "b"):
            for other, name in [(2, "spatial"), (3, "temporal")]:
                STATS[(config, band, k, name)] = contrast_stats(config, band, k, other)
                p, eff, sig, runs = STATS[(config, band, k, name)]
                print(f"{config}, {band}, {READ[k]}, identity vs {name}: pairs {len(TR[(config, band, k)]['pair'])} in"
                      f" {len(set(TR[(config, band, k)]['subject']))} subjects; FDR-significant points {int(sig.sum())}/91;"
                      f" runs (start ms, end ms, sign of identity - other, sustained) {runs}")

d = TR[("methods", "delta-theta", "a")]
for ti in (0, 45, 90):
    e2, F2, p2 = fcx3.mixedlm_check(d["z"][2][:, ti], d["z"][1][:, ti], d["subject"], d["pair"])
    e1, F1, p1, _ = fcx3.paired_lme(d["z"][2][:, ti], d["z"][1][:, ti], d["subject"])
    print(f"check at {TIMES[ti]} ms (main spectra, delta-theta, a, identity vs spatial): paired {e1:.3f}, F {F1:.2f}, p {p1:.3g}; MixedLM {e2:.3f}, F {F2:.2f}, p {p2:.3g}")

In [ ]:
for k in ("a", "b"):
    fig, axes = plt.subplots(2, 2, figsize=(12, 6), sharex=True)
    for i, band in enumerate(fcx3.BANDS):
        for j, (other, name) in enumerate([(2, "spatial"), (3, "temporal")]):
            ax = axes[i, j]
            for c, col, lab in [(1, "tab:blue", "identity"), (other, "tab:orange" if other == 2 else "tab:pink", name)]:
                m, se, _ = grand("methods", band, k, c)
                ax.plot(TIMES, m, color=col, label=lab)
                ax.fill_between(TIMES, m - se, m + se, color=col, alpha=0.2)
            sig = STATS[("methods", band, k, name)][2]
            ax.plot(TIMES[sig], np.full(sig.sum(), ax.get_ylim()[0]), "k|")
            ax.axhline(0, color="grey", lw=0.5)
            ax.set_title(f"{band}, identity vs {name}, reading ({k})")
            ax.legend(fontsize=7)
    axes[1, 0].set_xlabel("ms after processing start")
    axes[1, 1].set_xlabel("ms after processing start")
    plt.tight_layout()
    plt.show()
print("Main spectra. Grand mean +- SEM over subjects; black marks: FDR-significant time points. Axes autoscaled, every point shown.")

In [ ]:
EXP_EPOCH = {"spatial": [(0, 120), (470, 610)], "temporal": [(0, 170), (660, 860)]}
TOL = {"spatial": [(None, 12), (47, 61)], "temporal": [(None, 17), (66, 86)]}
for config in fcx3.PSI_T_CONFIGS:
    for k in ("a", "b"):
        print(f"{TCFG[config]}, {READ[k]}:")
        late_end = {}
        for name in ("spatial", "temporal"):
            runs = [r for r in STATS[(config, "delta-theta", k, name)][3] if r[3]]
            early = [r for r in runs if r[0] <= 10]
            late = [r for r in runs if r[0] > 10]
            print(f"  identity vs {name}: sustained runs {runs}; early (start within one step of 0): {early}; later: {late}")
            (e0, e1), (l0, l1) = EXP_EPOCH[name]
            (_, te1), (tl0, tl1) = TOL[name]
            ok_e = len(early) == 1 and abs(early[0][1] - e1) <= te1 and early[0][2] == 1
            ok_l = len(late) == 1 and abs(late[0][0] - l0) <= tl0 and abs(late[0][1] - l1) <= tl1 and late[0][2] == -1
            print(f"    one early epoch, positive, end within tolerance: {ok_e}; one late epoch, negative, bounds within tolerance: {ok_l}")
            late_end[name] = late
        if late_end["spatial"] and late_end["temporal"]:
            print(f"  gap spatial late end to temporal late start: {late_end['temporal'][0][0] - late_end['spatial'][0][1]} ms (expected 50 +- 20)")

**Result: not reproduced.** No time point differs between conditions after FDR in delta-theta
under either PSI reading or either spectra reading, so there is no sustained epoch. Under reading
(a) the selection holds 78 pairs in 7 subjects; under reading (b) it holds 1 pair (S1), on which no
test can be formed.

In [ ]:
EPOCHS = [("spatial", (0, 120), 2.62, ">"), ("temporal", (0, 170), 2.35, ">"), ("spatial", (470, 610), -3.04, "<"), ("temporal", (660, 860), -3.57, "<")]
ok = lambda v, exp, rel: (v > 1.96 and (v >= exp - 0.1 * abs(exp))) if rel == ">" else (v < -1.96 and (v <= exp + 0.1 * abs(exp)))
for config in fcx3.PSI_T_CONFIGS:
    for k in ("a", "b"):
        print(f"{TCFG[config]}, {READ[k]}:")
        mi, _, per = grand(config, "delta-theta", k, 1)
        d = TR[(config, "delta-theta", k)]
        for name, (a, b), exp, rel in EPOCHS:
            m = (TIMES >= a) & (TIMES <= b)
            other = 2 if name == "spatial" else 3
            # readings: (a) grand mean over subjects averaged over the epoch; (b) all pairs pooled; (c) least extreme point; (B) identity - other
            mo, _, _ = grand(config, "delta-theta", k, other)
            ra = mi[m].mean()
            rb = d["z"][1][:, m].mean()
            rc = mi[m].min() if rel == ">" else mi[m].max()
            rB = (mi - mo)[m].mean()
            print(f"  identity, {a}-{b} ms (expected {rel} {exp}): (a) {ra:.2f} {ok(ra, exp, rel)}, (b) pooled pairs {rb:.2f} {ok(rb, exp, rel)},"
                  f" (c) least extreme point {rc:.2f} {ok(rc, exp, rel)}, (B) identity - {name} {rB:.2f} {ok(rB, exp, rel)}")
        print("  per-subject identity mean z over the four epochs:")
        print("  ", {name + f" {a}-{b}": per.loc[:, (TIMES >= a) & (TIMES <= b)].mean(axis=1).round(2).to_dict() for name, (a, b), _, _ in EPOCHS})

**Result: not reproduced.** In the paper's four epochs, identity-trial delta-theta PSI z meets the
criterion under no reading of the averaging, of what is measured, of PSI or of the spectra. Under
the main spectra, reading (a), the grand means are 0.73, 0.60, 0.22 and -0.13 (paper > 2.62,
> 2.35, < -3.04, < -3.57); under reading (b), on S1's one pair, 1.31, 0.93, -1.98 and 0.13, where
the third has the right sign and passes -1.96 (-2.20 as identity minus spatial) but is outside the
bound (-3.04 within 10%). Every value of the other readings is below 1.96 in absolute value.

In [ ]:
for config in fcx3.PSI_T_CONFIGS:
    for k in ("a", "b"):
        for c, name in [(2, "spatial"), (3, "temporal")]:
            m, se, per = grand(config, "delta-theta", k, c)
            pooled = TR[(config, "delta-theta", k)]["z"][c].mean(0)
            print(f"{TCFG[config]}, {READ[k]}, {name}: grand mean max |z| {np.abs(m).max():.2f} at {TIMES[np.argmax(np.abs(m))]} ms;"
                  f" points with |z| >= 1.96: {int((np.abs(m) >= 1.96).sum())}/91; >= 1.60: {int((np.abs(m) >= 1.60).sum())};"
                  f" pooled pairs max |z| {np.abs(pooled).max():.2f}; per-subject max |z| {per.abs().max(axis=1).round(2).to_dict()}")
    # the same measure on identity trials, where the paper reports |z| above 2.35, as a check that the measure can show direction
    for k in ("a", "b"):
        m, _, _ = grand(config, "delta-theta", k, 1)
        print(f"{TCFG[config]}, {READ[k]}, identity (reference): grand mean max |z| {np.abs(m).max():.2f}; points with |z| >= 1.96: {int((np.abs(m) >= 1.96).sum())}/91")

**Result: not reproduced, resting on an assumption.** With PSI reading (b) adopted (see the
Summary), the selection of directional pairs leaves one pair (S1), and its spatial-trial
delta-theta PSI z reaches |z| = 3.07 (140 ms) and its temporal-trial z 2.36 (180 ms), with 16 and 11
of 91 time points at |z| >= 1.96 (paper, all |z| < 1.60). Its identity-trial z also passes 1.96 at
9 points, so on one pair such excursions are common. Under reading (a) (78 pairs in 7 subjects) the
grand means stay at or below 0.87 and 1.01, which would meet the criterion, but the identity-trial
grand mean also stays below 0.99, where the paper reports |z| above 2.35, so that absence shows
little.

In [ ]:
for config in fcx3.PSI_T_CONFIGS:
    for k in ("a", "b"):
        for name in ("spatial", "temporal"):
            p, eff, sig, runs = STATS[(config, "alpha-beta", k, name)]
            longest = max([r[1] - r[0] for r in runs], default=None)
            print(f"alpha-beta, {TCFG[config]}, {READ[k]}, identity vs {name}: FDR-significant points {int(sig.sum())}/91 (smallest p {p.min():.3g});"
                  f" runs {runs}; longest {longest} ms; sustained (>= 100 ms): {[r for r in runs if r[3]]}; (> 100 ms): {[r for r in runs if r[1] - r[0] > 100]}")

**Result: reproduced, resting on an assumption.** With PSI reading (b) adopted (74 pairs in 6
subjects), 30 (identity against spatial) and 35 (against temporal) of 91 time points differ after
FDR, and the longest run of one sign lasts 40 ms, so no difference is sustained under either
spectra reading (whole-epoch windows: 36 and 37 points, longest 50 and 40 ms). Under reading (a)
(104 pairs) no point is significant after FDR (smallest p 0.0015 and 0.0027), the criterion's
partial match.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| table1-electrodes | partially reproduced |
| trial-counts | reproduced |
| fig1b-accuracy | reproduced |
| fig1b-rt-identity-slower | reproduced |
| erp-no-condition-difference | not reproduced (assumption: neighbours adjacent by contact number, 0.05 per tail) |
| fig2-spatial-power | partially reproduced |
| fig3-temporal-power | partially reproduced |
| power-difficulty-subjects | reproduced |
| power-low-frequencies | reproduced (assumption: the 2-Hz end of the range counts as a local maximum) |
| fig4-maintenance-parietal-led | not reproduced |
| fig4a-delta-theta-shift | not reproduced |
| fig4c-alpha-beta-shift | partially reproduced (assumption: PSI reading (b)) |
| directional-pairs | not reproduced |
| fig5a-delta-theta-epochs | not reproduced |
| fig5a-identity-direction | not reproduced |
| fig5a-spatiotemporal-bidirectional | not reproduced (assumption: PSI reading (b)) |
| fig5b-alpha-beta-no-sustained | reproduced (assumption: PSI reading (b)) |

**How the analysis differs from the paper's.** Everything is computed in Python from the derived
epochs; FieldTrip itself was not run. Windows, trial selection and spectral parameters follow the
Methods and the deposited MATLAB scripts; where they differ, the Methods are the main reading and
the scripts a variant. (1) ERP and power segments are zero padded as the Methods say; the scripts
filter and transform the whole epoch, so that low-frequency windows take in the data around the
segment. (2) The PSI null shuffles the frequencies of one signal and recomputes the same statistic,
as the Methods say. psianalysis.m z-scores the mean of per-trial PSI against the slope of the
trial-mean spectrum, a different statistic, and on trial-shuffled data that null gives |z| > 1.96
in 90% of pairs; it is shown as a variant. With the Methods' null the trial-shuffled data give 6 to 7%
(reading (a)) and 0.1% (delta-theta) or 6% (alpha-beta) under reading (b), and the real data give
few directional pairs, so none of the large PSI effects of the paper appears. (3) The cluster test's
neighbours and alpha per tail are stated neither in the paper nor in the scripts, which hold no
cluster test: the adjacency of contact numbers and 0.05 per tail are adopted, and every reading is
shown. (4) PSI from trial-averaged cross-spectra (reading (a), Nolte et al. 2008) or per trial
(reading (b)): for claims where the readings give different outcomes, reading (b) is adopted,
because it is the structure of psianalysis.m and Figure 4A's legend gives values "per subject
across all trials". (5) The baseline z-score of power is computed in closed form; it cannot change
the condition t-tests. (6) The mixed models are computed as paired tests over pairs, which equal
them when every pair has both values (checked against statsmodels). As in the paper's degrees of
freedom, pairs that share electrodes are treated as independent, which inflates the pair-level F;
subject-level tests are printed beside them for the task-induced PSI.

**What was found about the data set.** The derived epochs, the continuous data and the trial
tables share one clock: every analysed channel's derived trace is a bipolar difference of two
contacts of the continuous data (r >= 0.9998), at fixed offsets from the table's columns that put the
scripts' windows where the Methods put them. The data set lists 187 analysed channels where the paper
reports 192 (S4 has 5 parietal channels, Table 1 10), and 1135 same-hemisphere pairs where the paper
reports 1129. The trial tables of S6 and S7 hold only the kept trials (101 and 90), unlike the
description. A timing contrast between trials with 900-ms and 1150-ms delays shows, where the
70-150 Hz response is clear (S2, S3, S4), that the responses in the processing epoch of the 1150-ms
trials occur 0.21 to 0.26 s later relative to the processing column than in the 900-ms trials, and
in S4 the same in the encoding epoch, while the pretrial epochs of these subjects show shifts of
0.03 s or less; this is consistent with the encoding and processing columns being about 0.25 s
early in 1150-ms trials, so that their processing window would include the end of the cue, but the
test does not prove it and is weak in the other subjects (S1, S5 and S6 show shifts of 0.28 to 0.35
s that do not pass the shuffle test). The ERP and power tests on the 900-ms-delay trials alone are
shown as variants. The deposited psi_compute.m does not run as written (an undefined `freq` and
`savDir`, and an electrode argument tested with `nargin > 2`), subtracterp.m selects temporal
trials by the delay column, which selects none, and psianalysis.m omits the conjugate of the
cross-spectrum and z-scores against a null that is not the statistic it tests.